# Transformer Decoder Demo (PyTorch)

# 1. Khởi tạo môi trường và Hyperparameters
Thiết lập các tham số mô hình dựa trên các ví dụ thực tế (như GPT-3 nhỏ hoặc cấu hình chuẩn)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Hyperparameters (Dựa trên cấu hình chuẩn trong tài liệu) [cite: 279, 288]
batch_size = 4
context_length = 8  # N [cite: 428]
d_model = 512       # d [cite: 271, 279]
num_heads = 8       # A [cite: 288]
d_k = d_model // num_heads # 64 [cite: 279]
d_ff = 2048         # d_ff [cite: 362]
vocab_size = 1000   # |V| [cite: 617]
num_layers = 6      # n_layer [cite: 411]

# 2. Cơ chế Multi-Head Attention với Masking
Đây là thành phần cốt lõi dùng để xây dựng các biểu diễn ngữ cảnh bằng cách tích hợp thông tin từ các token xung quanh. Chúng ta sử dụng Causal Masking để đảm bảo mô hình không "nhìn thấy" các từ trong tương lai khi training.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Các ma trận trọng số W^Q, W^K, W^V [cite: 210, 283]
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model) # Project về output d [cite: 230, 303]

    def forward(self, x):
        B, N, D = x.shape

        # 1. Generate Query, Key, Value [cite: 211, 212]
        q = self.W_q(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Scaled Dot-Product Attention [cite: 216, 219]
        # score = (Q * K^T) / sqrt(d_k) [cite: 219]
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 3. Causal Masking (Upper triangular to -inf) [cite: 464, 465]
        mask = torch.triu(torch.ones(N, N), diagonal=1).bool().to(x.device)
        attn_scores = attn_scores.masked_fill(mask, float('-inf'))

        # 4. Softmax & Weighted Sum [cite: 220, 221]
        attn_weights = F.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_weights, v) # [B, A, N, d_v]

        # 5. Concatenate & Linear [cite: 296, 303]
        output = output.transpose(1, 2).contiguous().view(B, N, D)
        return self.W_o(output)

# 3. Transformer Block (Residual Stream)
Mỗi khối Transformer gồm một lớp Multi-Head Attention và một lớp Feedforward, kết hợp với Residual Connections và Layer Normalization.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        # Sử dụng kiến trúc Prenorm (Layer Norm trước) [cite: 352, 418]
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(), # [cite: 363]
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        # Residual Stream: x = x + Attention(LN(x)) [cite: 388, 585]
        x = x + self.attn(self.ln1(x))
        # Residual Stream: x = x + FFN(LN(x)) [cite: 394, 586]
        x = x + self.ffn(self.ln2(x))
        return x

# 4. Mô hình Transformer Decoder hoàn chỉnh
Kết hợp Input Encoding (Token + Positional Embeddings) và Language Modeling Head.

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len):
        super().__init__()
        # Input Encoding [cite: 616, 654, 655]
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)

        # Stacked Blocks [cite: 411, 1029]
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model) # Extra layer norm cuối cùng [cite: 416, 421]

        # LM Head: Unembedding matrix U (thường tied với E) [cite: 708, 712]
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight # Weight Tying [cite: 708]

    def forward(self, idx):
        B, N = idx.shape
        # Tính toán embeddings [cite: 679]
        tok_emb = self.token_embedding(idx)
        pos_indices = torch.arange(N, device=idx.device)
        pos_emb = self.pos_embedding(pos_indices)

        x = tok_emb + pos_emb # Composite Embeddings [cite: 655, 659]

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x) # [B, N, V] [cite: 707, 736]
        return logits

# 5. Chạy Demo
Tạo dữ liệu giả lập và thực hiện forward pass để lấy xác suất token tiếp theo.

In [ ]:
# Khởi tạo mô hình
model = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, context_length)

# Giả lập input là indices của câu "Thanks for all the" [cite: 618, 619]
# Ví dụ: [5, 4000, 10532, 2224] -> ở đây dùng giá trị nhỏ hơn vocab_size
input_indices = torch.randint(0, vocab_size, (batch_size, context_length))

# Forward pass
logits = model(input_indices)

# Lấy xác suất của token cuối cùng để dự đoán token N+1 [cite: 705, 736]
next_token_logits = logits[:, -1, :]
probs = F.softmax(next_token_logits, dim=-1)

print(f"Hình dạng Logits: {logits.shape}") # [Batch, Context, Vocab]
print(f"Dự đoán token tiếp theo cho batch 1: {torch.argmax(probs[0]).item()}")

Hình dạng Logits: torch.Size([4, 8, 1000])
Dự đoán token tiếp theo cho batch 1: 85
